In [0]:
# Databricks notebook:
# 07_new_relic_pipeline_health

import json
import hashlib
from datetime import datetime, timezone

import requests

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)
from delta.tables import DeltaTable


# Make timestamp handling consistent.
spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)


# ---------------------------------------------------------
# PROJECT CONFIGURATION
# ---------------------------------------------------------

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"
VOLUME = "volume1"

NAMESPACE = f"{CATALOG}.{SCHEMA}"

PIPELINE_NAME = "cold_chain_databricks_pipeline"
TEAM_NAME = "cold-chain-capstone"
ENVIRONMENT = "training"


# ---------------------------------------------------------
# TABLE NAMES
# ---------------------------------------------------------

AUDIT_TABLE = (
    f"{NAMESPACE}.pipeline_run_audit"
)

BRONZE_SENSOR_TABLE = (
    f"{NAMESPACE}.bronze_sensor_events"
)

SILVER_SENSOR_TABLE = (
    f"{NAMESPACE}.silver_sensor_events"
)

QUARANTINE_SENSOR_TABLE = (
    f"{NAMESPACE}.quarantine_sensor_events"
)

ENRICHED_SENSOR_TABLE = (
    f"{NAMESPACE}.silver_sensor_enriched"
)

TEMPERATURE_WINDOWS_TABLE = (
    f"{NAMESPACE}.shipment_temperature_windows"
)

SHIPMENT_HEALTH_TABLE = (
    f"{NAMESPACE}.shipment_health"
)

DELIVERY_PERFORMANCE_TABLE = (
    f"{NAMESPACE}.delivery_performance"
)

TELEMETRY_LOG_TABLE = (
    f"{NAMESPACE}.new_relic_telemetry_log"
)


# ---------------------------------------------------------
# NOTEBOOK WIDGETS
# ---------------------------------------------------------

def create_text_widget_if_missing(
    widget_name,
    default_value
):
    try:
        dbutils.widgets.get(widget_name)
    except Exception:
        dbutils.widgets.text(
            widget_name,
            default_value
        )


def create_dropdown_widget_if_missing(
    widget_name,
    default_value,
    allowed_values
):
    try:
        dbutils.widgets.get(widget_name)
    except Exception:
        dbutils.widgets.dropdown(
            widget_name,
            default_value,
            allowed_values
        )


create_text_widget_if_missing(
    "run_id",
    ""
)

create_dropdown_widget_if_missing(
    "new_relic_region",
    "US",
    ["US", "EU", "JP"]
)

create_dropdown_widget_if_missing(
    "publish_enabled",
    "false",
    ["true", "false"]
)


REQUESTED_RUN_ID = (
    dbutils.widgets
    .get("run_id")
    .strip()
)

NEW_RELIC_REGION = (
    dbutils.widgets
    .get("new_relic_region")
    .strip()
    .upper()
)

PUBLISH_ENABLED = (
    dbutils.widgets
    .get("publish_enabled")
    .strip()
    .lower()
    == "true"
)


assert NEW_RELIC_REGION in {
    "US",
    "EU",
    "JP"
}, (
    "new_relic_region must be US, EU, or JP"
)


print("==============================================")
print("NEW RELIC PIPELINE HEALTH")
print("==============================================")
print("Namespace          :", NAMESPACE)
print("Pipeline           :", PIPELINE_NAME)
print("Team               :", TEAM_NAME)
print("Environment        :", ENVIRONMENT)
print("Requested run ID   :", REQUESTED_RUN_ID or "latest")
print("New Relic region   :", NEW_RELIC_REGION)
print("Publishing enabled :", PUBLISH_ENABLED)

In [0]:
# ---------------------------------------------------------
# VALIDATE REQUIRED TABLES
# ---------------------------------------------------------

required_tables = {
    "pipeline audit": AUDIT_TABLE,
    "Bronze sensors": BRONZE_SENSOR_TABLE,
    "Silver sensors": SILVER_SENSOR_TABLE,
    "sensor quarantine": QUARANTINE_SENSOR_TABLE,
    "enriched sensors": ENRICHED_SENSOR_TABLE,
    "temperature windows": TEMPERATURE_WINDOWS_TABLE,
    "shipment health": SHIPMENT_HEALTH_TABLE,
    "delivery performance": DELIVERY_PERFORMANCE_TABLE
}


print("==============================================")
print("REQUIRED TABLE CHECK")
print("==============================================")


for logical_name, table_name in required_tables.items():
    table_exists = spark.catalog.tableExists(
        table_name
    )

    print(
        f"{logical_name}:",
        "FOUND" if table_exists else "MISSING"
    )

    assert table_exists, (
        f"Required table is missing: {table_name}"
    )


# ---------------------------------------------------------
# READ AUDIT TABLE
# ---------------------------------------------------------

audit_df = spark.table(
    AUDIT_TABLE
)


required_audit_columns = {
    "run_id",
    "orchestrator",
    "airflow_dag_id",
    "airflow_run_id",
    "databricks_job_id",
    "databricks_run_id",
    "status",
    "started_at",
    "completed_at",
    "input_rows",
    "accepted_rows",
    "rejected_rows",
    "rejected_percent",
    "temperature_exception_count",
    "gold_temperature_window_rows",
    "shipment_health_rows",
    "delivery_performance_rows",
    "error_message"
}


missing_audit_columns = (
    required_audit_columns
    - set(audit_df.columns)
)


assert not missing_audit_columns, (
    "Audit table is missing these columns: "
    f"{sorted(missing_audit_columns)}"
)


# ---------------------------------------------------------
# SELECT REQUESTED OR LATEST COMPLETED RUN
# ---------------------------------------------------------

if REQUESTED_RUN_ID:
    selected_audit_df = (
        audit_df
        .filter(
            F.col("run_id")
            == REQUESTED_RUN_ID
        )
        .orderBy(
            F.col("completed_at")
            .desc_nulls_last()
        )
        .limit(1)
    )
else:
    selected_audit_df = (
        audit_df
        .filter(
            F.col("completed_at").isNotNull()
        )
        .orderBy(
            F.col("completed_at").desc()
        )
        .limit(1)
    )


selected_rows = selected_audit_df.collect()


assert len(selected_rows) == 1, (
    "No matching completed pipeline run was found. "
    "Run 06_operational_readiness first or provide "
    "a valid run_id."
)


audit_run = selected_rows[0]

RUN_ID = str(
    audit_run["run_id"]
)

started_at = audit_run["started_at"]
completed_at = audit_run["completed_at"]


assert started_at is not None, (
    "The selected audit record has no started_at value."
)

assert completed_at is not None, (
    "The selected audit record has no completed_at value."
)


print("==============================================")
print("SELECTED PIPELINE RUN")
print("==============================================")
print("Run ID       :", RUN_ID)
print("Status       :", audit_run["status"])
print("Started at   :", started_at)
print("Completed at :", completed_at)
print("Orchestrator :", audit_run["orchestrator"])

display(selected_audit_df)

In [0]:
# ---------------------------------------------------------
# READ DATABRICKS JOB CONTEXT
# ---------------------------------------------------------

def get_context_tag(tag_name):
    try:
        context = (
            dbutils.notebook
            .entry_point
            .getDbutils()
            .notebook()
            .getContext()
        )

        value = (
            context
            .tags()
            .apply(tag_name)
        )

        if value is None:
            return None

        value = str(value).strip()

        return value if value else None

    except Exception:
        return None


audit_job_id = (
    str(audit_run["databricks_job_id"])
    if audit_run["databricks_job_id"] is not None
    else None
)

audit_databricks_run_id = (
    str(audit_run["databricks_run_id"])
    if audit_run["databricks_run_id"] is not None
    else None
)


DATABRICKS_JOB_ID = (
    get_context_tag("jobId")
    or get_context_tag("job_id")
    or audit_job_id
)


DATABRICKS_RUN_ID = (
    get_context_tag("jobRunId")
    or get_context_tag("runId")
    or get_context_tag("job_run_id")
    or audit_databricks_run_id
)


DATABRICKS_TASK_KEY = (
    get_context_tag("taskKey")
    or get_context_tag("task_key")
)


print("Databricks Job ID :", DATABRICKS_JOB_ID)
print("Databricks Run ID :", DATABRICKS_RUN_ID)
print("Databricks task   :", DATABRICKS_TASK_KEY)


# ---------------------------------------------------------
# UPDATE EXISTING AUDIT RECORD
# ---------------------------------------------------------

job_context_schema = StructType([
    StructField(
        "run_id",
        StringType(),
        False
    ),
    StructField(
        "databricks_job_id",
        StringType(),
        True
    ),
    StructField(
        "databricks_run_id",
        StringType(),
        True
    )
])


job_context_df = spark.createDataFrame(
    [
        (
            RUN_ID,
            DATABRICKS_JOB_ID,
            DATABRICKS_RUN_ID
        )
    ],
    schema=job_context_schema
)


audit_delta = DeltaTable.forName(
    spark,
    AUDIT_TABLE
)


(
    audit_delta.alias("target")
    .merge(
        job_context_df.alias("source"),
        "target.run_id = source.run_id"
    )
    .whenMatchedUpdate(
        set={
            "databricks_job_id": """
                COALESCE(
                    source.databricks_job_id,
                    target.databricks_job_id
                )
            """,
            "databricks_run_id": """
                COALESCE(
                    source.databricks_run_id,
                    target.databricks_run_id
                )
            """
        }
    )
    .execute()
)


print(
    "Audit Job context updated for run:",
    RUN_ID
)

In [0]:
# ---------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------

def safe_int(value, default_value=0):
    if value is None:
        return default_value

    return int(value)


def safe_float(value, default_value=None):
    if value is None:
        return default_value

    return float(value)


def scope_to_selected_run(
    dataframe,
    logical_name
):
    columns = set(dataframe.columns)

    if "run_id" in columns:
        print(
            f"{logical_name}: scoped using run_id"
        )

        return dataframe.filter(
            F.col("run_id") == RUN_ID
        )

    if "_processing_time" in columns:
        print(
            f"{logical_name}: scoped using "
            "_processing_time"
        )

        return dataframe.filter(
            (
                F.col("_processing_time")
                >= F.lit(started_at)
            )
            & (
                F.col("_processing_time")
                <= F.lit(completed_at)
            )
        )

    print(
        f"WARNING: {logical_name} has no run_id or "
        "_processing_time. Supporting calculations "
        "will use the full table."
    )

    return dataframe


# ---------------------------------------------------------
# READ TABLES
# ---------------------------------------------------------

bronze_sensor_df = spark.table(
    BRONZE_SENSOR_TABLE
)

silver_sensor_df = spark.table(
    SILVER_SENSOR_TABLE
)

quarantine_sensor_df = spark.table(
    QUARANTINE_SENSOR_TABLE
)

enriched_sensor_df = spark.table(
    ENRICHED_SENSOR_TABLE
)

temperature_windows_df = spark.table(
    TEMPERATURE_WINDOWS_TABLE
)

shipment_health_df = spark.table(
    SHIPMENT_HEALTH_TABLE
)

delivery_performance_df = spark.table(
    DELIVERY_PERFORMANCE_TABLE
)


# ---------------------------------------------------------
# SCOPE SUPPORTING TABLES TO THE SELECTED RUN
# ---------------------------------------------------------

run_bronze_sensor_df = scope_to_selected_run(
    bronze_sensor_df,
    "Bronze sensors"
)

run_silver_sensor_df = scope_to_selected_run(
    silver_sensor_df,
    "Silver sensors"
)

run_quarantine_sensor_df = scope_to_selected_run(
    quarantine_sensor_df,
    "Quarantine sensors"
)

run_enriched_sensor_df = scope_to_selected_run(
    enriched_sensor_df,
    "Enriched sensors"
)

run_temperature_windows_df = scope_to_selected_run(
    temperature_windows_df,
    "Temperature windows"
)

run_shipment_health_df = scope_to_selected_run(
    shipment_health_df,
    "Shipment health"
)

run_delivery_performance_df = scope_to_selected_run(
    delivery_performance_df,
    "Delivery performance"
)


# ---------------------------------------------------------
# USE AUDIT TABLE FOR RUN-LEVEL COUNTS
# ---------------------------------------------------------

input_rows = safe_int(
    audit_run["input_rows"]
)

accepted_rows = safe_int(
    audit_run["accepted_rows"]
)

rejected_rows = safe_int(
    audit_run["rejected_rows"]
)

output_rows = accepted_rows


audit_rejected_percent = safe_float(
    audit_run["rejected_percent"]
)


if audit_rejected_percent is not None:
    rejected_percent = audit_rejected_percent
elif input_rows > 0:
    rejected_percent = float(
        rejected_rows / input_rows * 100.0
    )
else:
    rejected_percent = None


temperature_exception_count = safe_int(
    audit_run["temperature_exception_count"]
)

gold_temperature_window_rows = safe_int(
    audit_run["gold_temperature_window_rows"]
)

shipment_health_rows = safe_int(
    audit_run["shipment_health_rows"]
)

delivery_performance_rows = safe_int(
    audit_run["delivery_performance_rows"]
)


gold_output_rows = (
    gold_temperature_window_rows
    + shipment_health_rows
    + delivery_performance_rows
)


no_data_condition = (
    input_rows == 0
)


# ---------------------------------------------------------
# CALCULATE DUPLICATES
# ---------------------------------------------------------

duplicate_rows = 0


if "event_id" in run_bronze_sensor_df.columns:
    duplicate_result = (
        run_bronze_sensor_df
        .filter(
            F.col("event_id").isNotNull()
        )
        .groupBy("event_id")
        .count()
        .filter(
            F.col("count") > 1
        )
        .select(
            F.sum(
                F.col("count") - F.lit(1)
            ).alias("duplicate_rows")
        )
        .first()
    )

    if duplicate_result is not None:
        duplicate_rows = int(
            duplicate_result["duplicate_rows"] or 0
        )
else:
    print(
        "WARNING: event_id is unavailable. "
        "duplicateRows will be reported as 0."
    )


duplicate_percent = (
    float(
        duplicate_rows / input_rows * 100.0
    )
    if input_rows > 0
    else None
)


# ---------------------------------------------------------
# CALCULATE BUSINESS-HEALTH COUNTS
# ---------------------------------------------------------

missing_sensor_count = 0


if (
    "missing_sensor_status"
    in run_shipment_health_df.columns
):
    missing_sensor_count = (
        run_shipment_health_df
        .filter(
            F.col("missing_sensor_status").isin(
                "MISSING",
                "STALE"
            )
        )
        .count()
    )
else:
    print(
        "WARNING: missing_sensor_status is unavailable."
    )


open_overdue_count = 0


if (
    "open_overdue_status"
    in run_delivery_performance_df.columns
):
    open_overdue_count = (
        run_delivery_performance_df
        .filter(
            F.col("open_overdue_status")
            == "OPEN_OVERDUE"
        )
        .count()
    )
else:
    print(
        "WARNING: open_overdue_status is unavailable."
    )


# ---------------------------------------------------------
# CALCULATE DURATION AND FRESHNESS
# ---------------------------------------------------------

duration_seconds = float(
    (
        completed_at - started_at
    ).total_seconds()
)


assert duration_seconds >= 0, (
    "Pipeline duration cannot be negative."
)


latest_event_timestamp = None


if "event_time" in run_enriched_sensor_df.columns:
    latest_event_row = (
        run_enriched_sensor_df
        .select(
            F.expr(
                "try_cast(event_time AS TIMESTAMP)"
            ).alias("parsed_event_time")
        )
        .filter(
            F.col("parsed_event_time").isNotNull()
        )
        .agg(
            F.max("parsed_event_time").alias(
                "latest_event_timestamp"
            )
        )
        .first()
    )

    if latest_event_row is not None:
        latest_event_timestamp = (
            latest_event_row[
                "latest_event_timestamp"
            ]
        )


latest_processing_timestamp = None


if "_processing_time" in run_bronze_sensor_df.columns:
    latest_processing_row = (
        run_bronze_sensor_df
        .agg(
            F.max("_processing_time").alias(
                "latest_processing_timestamp"
            )
        )
        .first()
    )

    if latest_processing_row is not None:
        latest_processing_timestamp = (
            latest_processing_row[
                "latest_processing_timestamp"
            ]
        )


freshness_seconds = None


if latest_event_timestamp is not None:
    freshness_seconds = max(
        0.0,
        float(
            (
                completed_at
                - latest_event_timestamp
            ).total_seconds()
        )
    )


source_to_processing_delay_seconds = None


if {
    "event_time",
    "_processing_time"
}.issubset(
    set(run_bronze_sensor_df.columns)
):
    delay_row = (
        run_bronze_sensor_df
        .withColumn(
            "parsed_event_time",
            F.expr(
                "try_cast(event_time AS TIMESTAMP)"
            )
        )
        .filter(
            F.col("parsed_event_time").isNotNull()
            & F.col("_processing_time").isNotNull()
        )
        .withColumn(
            "delay_seconds",
            F.unix_timestamp("_processing_time")
            - F.unix_timestamp(
                "parsed_event_time"
            )
        )
        .filter(
            F.col("delay_seconds") >= 0
        )
        .agg(
            F.max("delay_seconds").alias(
                "maximum_delay_seconds"
            )
        )
        .first()
    )

    if (
        delay_row is not None
        and delay_row["maximum_delay_seconds"]
        is not None
    ):
        source_to_processing_delay_seconds = float(
            delay_row["maximum_delay_seconds"]
        )


# ---------------------------------------------------------
# REJECTION REASON BREAKDOWN
# ---------------------------------------------------------

rejection_reason_counts = {}


if (
    "rejection_reason"
    in run_quarantine_sensor_df.columns
):
    rejection_reason_rows = (
        run_quarantine_sensor_df
        .filter(
            F.col("rejection_reason").isNotNull()
        )
        .groupBy("rejection_reason")
        .count()
        .orderBy("rejection_reason")
        .collect()
    )

    rejection_reason_counts = {
        str(row["rejection_reason"]): int(row["count"])
        for row in rejection_reason_rows
    }


mandatory_field_reasons = {
    "missing_event_id",
    "missing_shipment_id"
}

invalid_data_type_reasons = {
    "nonnumeric_temperature",
    "invalid_event_time"
}

reference_validation_reasons = {
    "unknown_shipment",
    "unknown_product"
}


mandatory_field_failures = sum(
    count_value
    for reason, count_value
    in rejection_reason_counts.items()
    if reason in mandatory_field_reasons
)


invalid_data_type_failures = sum(
    count_value
    for reason, count_value
    in rejection_reason_counts.items()
    if reason in invalid_data_type_reasons
)


reference_validation_failures = sum(
    count_value
    for reason, count_value
    in rejection_reason_counts.items()
    if reason in reference_validation_reasons
)


schema_validation_failures = 0


# ---------------------------------------------------------
# DISPLAY CALCULATED METRICS
# ---------------------------------------------------------

print("==============================================")
print("ACTUAL PIPELINE HEALTH METRICS")
print("==============================================")
print("Run ID                    :", RUN_ID)
print("Pipeline status           :", audit_run["status"])
print("Duration seconds          :", duration_seconds)
print("Input rows                :", input_rows)
print("Accepted rows             :", accepted_rows)
print("Rejected rows             :", rejected_rows)
print("Duplicate rows            :", duplicate_rows)

In [0]:
# =========================================================
# CELL 5: BUILD NEW RELIC EVENTS AND PAYLOAD
# =========================================================

def iso_utc(value):
    """
    Convert a Python datetime value to an ISO-formatted
    UTC string for New Relic.
    """
    if value is None:
        return None

    if value.tzinfo is None:
        value = value.replace(
            tzinfo=timezone.utc
        )

    return value.astimezone(
        timezone.utc
    ).isoformat()


def remove_none_values(event):
    """
    Remove attributes whose values are None because
    New Relic custom-event attributes should contain
    usable scalar values.
    """
    return {
        key: value
        for key, value in event.items()
        if value is not None
    }


# ---------------------------------------------------------
# EVENT 1: PIPELINE EXECUTION HEALTH
# ---------------------------------------------------------

pipeline_event = {
    "eventType": "ColdChainPipelineRun",

    "team": TEAM_NAME,
    "pipeline": PIPELINE_NAME,
    "environment": ENVIRONMENT,

    "runId": RUN_ID,

    "orchestrator": (
        audit_run["orchestrator"]
        or "databricks"
    ),

    "airflowDagId": (
        audit_run["airflow_dag_id"]
    ),

    "airflowRunId": (
        audit_run["airflow_run_id"]
    ),

    "databricksJobId": (
        DATABRICKS_JOB_ID
    ),

    "databricksRunId": (
        DATABRICKS_RUN_ID
    ),

    "databricksTaskKey": (
        DATABRICKS_TASK_KEY
    ),

    "status": (
        audit_run["status"]
    ),

    "startedAt": (
        iso_utc(started_at)
    ),

    "completedAt": (
        iso_utc(completed_at)
    ),

    "durationSeconds": (
        duration_seconds
    ),

    "inputRows": (
        input_rows
    ),

    "outputRows": (
        output_rows
    ),

    "acceptedRows": (
        accepted_rows
    ),

    "rejectedRows": (
        rejected_rows
    ),

    "duplicateRows": (
        duplicate_rows
    ),

    "rejectedPercent": (
        rejected_percent
    ),

    "duplicatePercent": (
        duplicate_percent
    ),

    "latestEventTimestamp": (
        iso_utc(latest_event_timestamp)
    ),

    "latestProcessingTimestamp": (
        iso_utc(latest_processing_timestamp)
    ),

    "freshnessSeconds": (
        freshness_seconds
    ),

    "sourceToProcessingDelaySeconds": (
        source_to_processing_delay_seconds
    ),

    "noDataCondition": (
        no_data_condition
    ),

    "temperatureExceptionCount": (
        temperature_exception_count
    ),

    "missingSensorCount": (
        missing_sensor_count
    ),

    "openOverdueCount": (
        open_overdue_count
    ),

    "bronzeSensorRows": (
        input_rows
    ),

    "silverAcceptedRows": (
        accepted_rows
    ),

    "silverRejectedRows": (
        rejected_rows
    ),

    "quarantineRows": (
        rejected_rows
    ),

    "goldTemperatureWindowRows": (
        gold_temperature_window_rows
    ),

    "shipmentHealthRows": (
        shipment_health_rows
    ),

    "deliveryPerformanceRows": (
        delivery_performance_rows
    ),

    "goldOutputRows": (
        gold_output_rows
    ),

    "businessKpiGenerationStatus": (
        "SUCCESS"
        if audit_run["status"] == "SUCCESS"
        else "FAILED"
    ),

    "goldRefreshTime": (
        iso_utc(completed_at)
    ),

    "errorMessage": (
        audit_run["error_message"]
    )
}


# ---------------------------------------------------------
# EVENT 2: DATA QUALITY
# ---------------------------------------------------------

data_quality_event = {
    "eventType": "ColdChainDataQuality",

    "team": TEAM_NAME,
    "pipeline": PIPELINE_NAME,
    "environment": ENVIRONMENT,

    "runId": RUN_ID,

    "databricksJobId": (
        DATABRICKS_JOB_ID
    ),

    "databricksRunId": (
        DATABRICKS_RUN_ID
    ),

    "inputRows": (
        input_rows
    ),

    "acceptedRows": (
        accepted_rows
    ),

    "rejectedRows": (
        rejected_rows
    ),

    "duplicateRows": (
        duplicate_rows
    ),

    "rejectedPercent": (
        rejected_percent
    ),

    "duplicatePercent": (
        duplicate_percent
    ),

    "schemaValidationFailures": (
        schema_validation_failures
    ),

    "mandatoryFieldFailures": (
        mandatory_field_failures
    ),

    "referenceValidationFailures": (
        reference_validation_failures
    ),

    "invalidDataTypeFailures": (
        invalid_data_type_failures
    )
}


# ---------------------------------------------------------
# EVENT 3: BUSINESS HEALTH
# ---------------------------------------------------------

business_health_event = {
    "eventType": "ColdChainBusinessHealth",

    "team": TEAM_NAME,
    "pipeline": PIPELINE_NAME,
    "environment": ENVIRONMENT,

    "runId": RUN_ID,

    "databricksJobId": (
        DATABRICKS_JOB_ID
    ),

    "databricksRunId": (
        DATABRICKS_RUN_ID
    ),

    "temperatureExceptionCount": (
        temperature_exception_count
    ),

    "missingSensorCount": (
        missing_sensor_count
    ),

    "openOverdueCount": (
        open_overdue_count
    ),

    "shipmentHealthRows": (
        shipment_health_rows
    ),

    "deliveryPerformanceRows": (
        delivery_performance_rows
    ),

    "evaluatedAt": (
        iso_utc(completed_at)
    )
}


# ---------------------------------------------------------
# EVENT 4: ONE EVENT PER REJECTION REASON
# ---------------------------------------------------------

rejection_events = []


for reason, count_value in (
    rejection_reason_counts.items()
):
    rejection_event = {
        "eventType": "ColdChainRejectionReason",

        "team": TEAM_NAME,
        "pipeline": PIPELINE_NAME,
        "environment": ENVIRONMENT,

        "runId": RUN_ID,

        "databricksJobId": (
            DATABRICKS_JOB_ID
        ),

        "databricksRunId": (
            DATABRICKS_RUN_ID
        ),

        "dataset": "sensor_events",

        "rejectionReason": (
            str(reason)
        ),

        "rejectedRows": (
            int(count_value)
        ),

        "inputRows": (
            input_rows
        )
    }

    rejection_events.append(
        rejection_event
    )


# ---------------------------------------------------------
# COMBINE ALL EVENTS
# ---------------------------------------------------------

new_relic_events = [
    pipeline_event,
    data_quality_event,
    business_health_event
] + rejection_events


# Remove attributes containing None.
new_relic_events = [
    remove_none_values(event)
    for event in new_relic_events
]


assert len(new_relic_events) >= 3, (
    "At least three New Relic events must be created."
)


# ---------------------------------------------------------
# CREATE FINAL JSON PAYLOAD
# ---------------------------------------------------------

payload_json = json.dumps(
    new_relic_events,
    sort_keys=True,
    separators=(",", ":"),
    default=str
)


print("==============================================")
print("NEW RELIC TELEMETRY PAYLOAD")
print("==============================================")
print("Events prepared:", len(new_relic_events))
print()


# This prints telemetry only.
# It does not print the New Relic license key.
print(
    json.dumps(
        new_relic_events,
        indent=2,
        default=str
    )
)